# Experiment 4: LSI (Latent Semantic Indexing) Topic Modeling
# Axis 1 - Method 4
## Hypothesis:
H1 - LSI uses the TF-IDF matrix. High TF-IDF words provide key information to tell different customer ticket topics apart.

H2 - LSI compresses high-dimensional text into a smaller latent space. We assume this compressed representation keeps the main thematic structure of the data.

H3 - LSI is a linear model and cannot handle complex semantics or word polysemy well. It may struggle to distinguish tickets with similar words but different actual problems.
## Pipeline:

**load data - TfidfVectorizer - LSI (TruncatedSVD) - topic export**

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [2]:
PROJECT_ROOT = Path.cwd().parents[2]
axis2_dir = PROJECT_ROOT / 'notebooks' / '03_Topic_and_Insights'
sys.path.insert(0, str(axis2_dir))
from Data_preprocessing import Parameters_Path as config

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'preprocessed_input.csv'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'Experiment_3_LSI'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LSI_TOPICS_PATH = RESULTS_DIR / 'Experiment_3_lsi_topics.json'
DOC_TOPIC_PATH = RESULTS_DIR / 'Experiment_3_lsi_doc_topics.csv'

tfidf_cfg = config.PARAMETERS['tfidf_vectorizer']
lsi_cfg = config.PARAMETERS['nmf']
N_COMPONENTS = int(lsi_cfg['n_components'])
N_TOP_WORDS = int(lsi_cfg['n_top_words'])
RANDOM_STATE = int(lsi_cfg['random_state'])

df = pd.read_csv(INPUT_PATH)
texts = df['text_cleaned_axis1'].astype(str).str.strip()
texts = texts[texts.ne('')]

vectorizer = TfidfVectorizer(
    max_df=tfidf_cfg['max_df'],
    min_df=tfidf_cfg['min_df'],
    max_features=tfidf_cfg['max_features'],
    stop_words=tfidf_cfg['stop_words']
)
X_tfidf = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

# LSI = TruncatedSVD

In [3]:
lsi = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=RANDOM_STATE
)
doc_topic = lsi.fit_transform(X_tfidf)

def extract_topics(model, vocab, n_top_words=8):
    topics = {}
    for i, topic in enumerate(model.components_, start=1):
        idx = topic.argsort()[-n_top_words:][::-1]
        topics[f'topic_{i}'] = {
            'keywords': [vocab[j] for j in idx],
            'weights': topic[idx].tolist()
        }
    return topics

lsi_topics = extract_topics(lsi, feature_names, N_TOP_WORDS)

with open(LSI_TOPICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(lsi_topics, f, indent=2, ensure_ascii=False)

# Normalization and alignment with LDA/NMF

In [4]:
doc_topic_abs = np.abs(doc_topic)
doc_topic_norm = doc_topic_abs / (doc_topic_abs.sum(axis=1, keepdims=True) + 1e-8)

topic_cols = [f'lsi_topic_{i}' for i in range(1, N_COMPONENTS + 1)]
doc_topic_df = pd.DataFrame(doc_topic_norm, columns=topic_cols)
doc_topic_df['lsi_dominant_topic'] = doc_topic_df[topic_cols].values.argmax(axis=1) + 1
doc_topic_df['max_topic_prob'] = doc_topic_df[topic_cols].values.max(axis=1)
doc_topic_df.to_csv(DOC_TOPIC_PATH, index=False, encoding='utf-8')

print("LSI topics preview:")
for topic_name, content in lsi_topics.items():
    print(f"{topic_name}: {', '.join(content['keywords'])}")

LSI topics preview:
topic_1: account, inquiry, software, data, refund, cancellation, billing, technical
topic_2: account, error, access, says, screen, peculiar, popping, mean
topic_3: started, screen, message, popping, peculiar, mean, says, error
topic_4: action, perform, desired, unable, option, guide, updates, software
